In [323]:
import torch, json
import numpy as np
import pandas as pd
import torch.nn as nn
import matplotlib.pyplot as plt

from PIL import Image
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import InterpolationMode

In [353]:
augmentation_pipeline_train = transforms.Compose([
    transforms.Resize((224, 224), interpolation=InterpolationMode.BILINEAR, antialias=False),
    transforms.RandomApply(  
        [transforms.ColorJitter(
            brightness=(0.8, 1.2),  # Brightness range: [0.8, 1.2]
            contrast=(0.8, 1.2)     # Contrast range: [0.8, 1.2]
        )],
        p=0.05                      # 5% of probability
    ),
    transforms.RandomHorizontalFlip(p=0.05),        
    transforms.RandomApply( 
        # Rotation range: [3 - 45]
        [transforms.RandomRotation(degrees=(3, 45))],
        p=0.05
    ),
    transforms.ToTensor()
])

augmentation_pipeline_val = transforms.Compose([
    transforms.Resize((224, 224), interpolation=InterpolationMode.BILINEAR, antialias=False),
    transforms.ToTensor()
])

In [359]:
img = augmentation_pipeline_train(Image.open('wiki_crop/00/10049200_1891-09-16_1958.jpg'))
img.shape

C:\Users\yerda\AppData\Roaming\Python\Python311\site-packages\torchvision\transforms\functional.py:488: UserWarning: Anti-alias option is always applied for PIL Image input. Argument antialias is ignored.
  warnings.warn("Anti-alias option is always applied for PIL Image input. Argument antialias is ignored.")


torch.Size([3, 224, 224])

In [362]:
df = pd.read_csv('wiki_imdb_lfw.csv')
df.shape

(536284, 8)

In [363]:
# Name encoder and decoder
encoder = {name:i for i, name in enumerate(df.name.unique())}
decoder = {indx:name for name, indx in encoder.items()}

# with open('encoder.json', 'w', encoding='utf-8') as f:
#     f.write(json.dumps(encoder))

# with open('decoder.json', 'w', encoding='utf-8') as f:
#     f.write(json.dumps(decoder))

In [364]:
with open('encoder.json', 'r', encoding='utf-8') as f:
    encoder = json.load(f)

with open('decoder.json', 'r', encoding='utf-8') as f:
    decoder = json.load(f)

In [382]:
class dataset(Dataset):
    def __init__(self, df, augmentation_pipeline, device):
        super(dataset, self).__init__()
        self.df = df
        self.device = device
        self.augmentation_pipeline = augmentation_pipeline
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, x):
        path = self.df.path[x]
        name = self.df.name[x]
        name = encoder[name]
        name = torch.tensor([name], dtype=torch.int32, device=self.device)
        
        img = Image.open(path).convert('RGB')
        img = self.augmentation_pipeline(img)
        img = img.to(self.device)
        return img, name

In [384]:
img, label = next(iter(dataset(df, augmentation_pipeline_train, device='cuda')))
img.shape, label.shape

C:\Users\yerda\AppData\Roaming\Python\Python311\site-packages\torchvision\transforms\functional.py:488: UserWarning: Anti-alias option is always applied for PIL Image input. Argument antialias is ignored.
  warnings.warn("Anti-alias option is always applied for PIL Image input. Argument antialias is ignored.")


(torch.Size([3, 224, 224]), torch.Size([1]))

In [386]:
dataLoader = DataLoader(dataset(df, augmentation_pipeline_train, device='cuda'), batch_size=32, shuffle=True)

In [388]:
img, label = next(iter(dataLoader))
img.shape, label.shape

C:\Users\yerda\AppData\Roaming\Python\Python311\site-packages\torchvision\transforms\functional.py:488: UserWarning: Anti-alias option is always applied for PIL Image input. Argument antialias is ignored.
  warnings.warn("Anti-alias option is always applied for PIL Image input. Argument antialias is ignored.")


(torch.Size([32, 3, 224, 224]), torch.Size([32, 1]))